# 🔍 Netfinder – Data Exploration
Euro 2024 | StatsBomb Open Data

Ziel: Daten kennenlernen, Schüsse analysieren, Freeze-Frames visualisieren

## 0. Setup

In [ ]:
# !pip install statsbombpy pandas matplotlib seaborn numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from statsbombpy import sb

# statsbombpy flattened column names (neue API)
# shot['shot']['body_part']['name'] → shot['shot_body_part']
# shot['shot']['outcome']['name']   → shot['shot_outcome']
# shot['shot']['statsbomb_xg']      → shot['shot_statsbomb_xg']
# shot['shot']['type']['name']      → shot['shot_type']
# shot['shot']['freeze_frame']      → shot['shot_freeze_frame']

pd.set_option('display.max_columns', None)
print('✅ Setup complete')

## 1. Daten laden

In [ ]:
# Alle verfügbaren Wettbewerbe
competitions = sb.competitions()
print(f"Verfügbare Wettbewerbe: {len(competitions)}")
competitions.head(10)

In [ ]:
# Euro 2024 Spiele
matches = sb.matches(competition_id=55, season_id=282)
print(f"Spiele gesamt: {len(matches)}")
matches[['match_id', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)

In [ ]:
# Ein einzelnes Spiel laden
match_id = matches.match_id.iloc[0]
match_info = matches[matches.match_id == match_id].iloc[0]
print(f"Spiel: {match_info['home_team']} vs {match_info['away_team']}")
print(f"Ergebnis: {match_info['home_score']}:{match_info['away_score']}")

events = sb.events(match_id=match_id)
print(f"\nEvents gesamt: {len(events)}")
print(f"Event-Typen: {events['type'].value_counts().to_dict()}")

## 2. Schüsse analysieren

In [ ]:
shots = events[events['type'] == 'Shot'].copy()
print(f"Schüsse in diesem Spiel: {len(shots)}")
shots.head(3)

In [ ]:
# Einen einzelnen Schuss genau anschauen
shot = shots.iloc[0]
print("=== Schuss-Details ===")
print(f"Spieler:     {shot['player']}")
print(f"Position:    {shot['location']}")
print(f"Körperteil:  {shot['shot_body_part']}")
print(f"Typ:         {shot['shot_type']}")
print(f"Ergebnis:    {shot['shot_outcome']}")
print(f"StatsBomb xG:{shot['shot_statsbomb_xg']}")
print(f"\nFreeze-Frame ({len(shot['shot_freeze_frame'])} Spieler):")
for p in shot['shot_freeze_frame'][:4]:
    rolle = 'Mitspieler' if p['teammate'] else 'Gegner'
    print(f"  {rolle}: {p['location']}")

In [ ]:
# Alle Schüsse aus allen Spielen laden (dauert ~1-2 min)
print("Lade alle Schüsse... (bitte warten)")
all_shots = []
for mid in matches.match_id:
    evs = sb.events(match_id=mid)
    s = evs[evs['type'] == 'Shot']
    all_shots.append(s)

all_shots_df = pd.concat(all_shots, ignore_index=True)
print(f"✅ Schüsse gesamt: {len(all_shots_df)}")
print(f"   davon Tore: {(all_shots_df['shot_outcome'] == 'Goal').sum()}")

## 3. Feature Engineering (Basis)

In [ ]:
# Feldmaße: 105m x 68m, Tor bei x=120, y=40
GOAL_X, GOAL_Y = 120, 40

def extract_basic_features(shot):
    x, y = shot['location']

    distance = np.sqrt((x - GOAL_X)**2 + (y - GOAL_Y)**2)
    angle    = np.arctan2(abs(y - GOAL_Y), abs(x - GOAL_X))
    is_goal  = 1 if shot['shot_outcome'] == 'Goal' else 0

    return {
        'x': x, 'y': y,
        'distance':     distance,
        'angle':        angle,
        'body_part':    shot['shot_body_part'],
        'shot_type':    shot['shot_type'],
        'statsbomb_xg': shot['shot_statsbomb_xg'],
        'is_goal':      is_goal
    }

features_df = pd.DataFrame(all_shots_df.apply(extract_basic_features, axis=1).tolist())
print(features_df.describe())

## 4. Visualisierungen

In [ ]:
def draw_half_pitch(ax):
    ax.set_facecolor('#2d5a1b')
    ax.set_xlim(60, 125)
    ax.set_ylim(0, 80)
    ax.add_patch(patches.Rectangle((102, 18), 18, 44, fill=False, color='white', lw=1.5))
    ax.add_patch(patches.Rectangle((114, 30), 6,  20, fill=False, color='white', lw=1.5))
    ax.add_patch(patches.Rectangle((120, 36), 2,  8,  fill=False, color='white', lw=2))
    ax.plot(108, 40, 'wo', markersize=3)
    ax.add_patch(patches.Arc((108, 40), 18.3, 18.3, angle=0, theta1=310, theta2=50, color='white', lw=1.5))
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
# Shot Map
fig, ax = plt.subplots(figsize=(10, 7))
draw_half_pitch(ax)

goals    = features_df[features_df['is_goal'] == 1]
no_goals = features_df[features_df['is_goal'] == 0]

ax.scatter(no_goals['x'], no_goals['y'], c='lightblue', alpha=0.4, s=30, label='Kein Tor')
ax.scatter(goals['x'],    goals['y'],    c='red',       alpha=0.9, s=80, zorder=5, label='Tor')
ax.legend(loc='upper left', facecolor='white')
ax.set_title(f'Euro 2024 – Shot Map ({len(features_df)} Schüsse, {len(goals)} Tore)',
             color='white', fontsize=13, pad=10)
fig.patch.set_facecolor('#1a1a2e')
plt.tight_layout()
plt.show()

In [ ]:
# Torrate nach Distanz & Winkel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, color, label in [
    (axes[0], 'distance', 'steelblue', 'Distanz (m)'),
    (axes[1], 'angle',    'coral',     'Winkel (rad)'),
]:
    bins = pd.cut(features_df[col], bins=10)
    rate = features_df.groupby(bins)[col if col != 'distance' else 'is_goal'].mean() \
           if col == 'angle' else features_df.groupby(bins)['is_goal'].mean()
    ax.bar(range(len(rate)), rate.values, color=color, alpha=0.8)
    ax.set_xticks(range(len(rate)))
    ax.set_xticklabels([str(b) for b in rate.index], rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Torrate nach {label.split(" ")[0]}')
    ax.set_ylabel('Torrate')
    ax.set_xlabel(label)

plt.tight_layout()
plt.show()

In [ ]:
# Freeze-Frame visualisieren
from matplotlib.lines import Line2D

def plot_freeze_frame(shot, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))
    draw_half_pitch(ax)

    freeze_frame = shot['shot_freeze_frame']
    sx, sy = shot['location']

    for player in freeze_frame:
        px, py = player['location']
        color  = '#00bfff' if player['teammate'] else '#ff4444'
        ax.plot(px, py, 'o', color=color, markersize=12, zorder=4)
        if player.get('keeper', False):
            ax.plot(px, py, 's', color='yellow', markersize=14, zorder=3)

    ax.plot(sx, sy, '*', color='white', markersize=20, zorder=5)
    ax.plot([sx, GOAL_X], [sy, GOAL_Y], '--', color='white', alpha=0.5, lw=1)

    xg = shot['shot_statsbomb_xg']
    title = f"{shot['player']} | {shot['shot_outcome']} | xG: {xg:.2f}" if pd.notna(xg) else f"{shot['player']} | {shot['shot_outcome']}"
    ax.set_title(title, color='white', fontsize=11)

    legend = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#00bfff', label='Mitspieler', markersize=10),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#ff4444', label='Gegner',     markersize=10),
        Line2D([0],[0], marker='*', color='w', markerfacecolor='white',   label='Schütze',    markersize=12),
    ]
    ax.legend(handles=legend, loc='upper left', facecolor='#1a1a2e', labelcolor='white')

goal_shots    = all_shots_df[all_shots_df['shot_outcome'] == 'Goal']
no_goal_shots = all_shots_df[all_shots_df['shot_outcome'] != 'Goal']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#1a1a2e')
plot_freeze_frame(goal_shots.iloc[0],    axes[0])
plot_freeze_frame(no_goal_shots.iloc[0], axes[1])
plt.suptitle('Freeze-Frame Vergleich', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Freeze-Frame → 3-Channel Heatmap (Vorschau auf features.py)
def freeze_frame_to_heatmap(shot, grid_size=(68, 52)):
    """
    Channel 0: Mitspieler
    Channel 1: Gegner
    Channel 2: Schütze
    """
    h, w = grid_size
    heatmap = np.zeros((3, h, w))

    def to_grid(x, y):
        gx = int(np.clip((x - 60) / 60 * w, 0, w - 1))
        gy = int(np.clip(y / 68 * h, 0, h - 1))
        return gy, gx

    for player in shot['shot_freeze_frame']:
        gy, gx = to_grid(*player['location'])
        channel = 0 if player['teammate'] else 1
        heatmap[channel, gy, gx] = 1.0

    gy, gx = to_grid(*shot['location'])
    heatmap[2, gy, gx] = 1.0

    return heatmap

heatmap = freeze_frame_to_heatmap(goal_shots.iloc[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, title, cmap) in enumerate(zip(
    axes,
    ['Channel 0: Mitspieler', 'Channel 1: Gegner', 'Channel 2: Schütze'],
    ['Blues', 'Reds', 'Greens']
)):
    ax.imshow(heatmap[i], cmap=cmap, origin='lower', aspect='auto')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Freeze-Frame als CNN Input (3 Channels)', fontsize=13)
plt.tight_layout()
plt.show()
print(f"Heatmap Shape: {heatmap.shape}  →  (channels, height, width)")

## 5. Quick Stats

In [ ]:
print("=== Euro 2024 – Shot Stats ===")
print(f"Schüsse gesamt:    {len(features_df)}")
print(f"Tore:              {features_df['is_goal'].sum()}")
print(f"Conversion Rate:   {features_df['is_goal'].mean():.1%}")
print(f"Ø Distanz:         {features_df['distance'].mean():.1f}m")
print(f"Ø StatsBomb xG:    {features_df['statsbomb_xg'].mean():.3f}")
print()
print("Torrate nach Körperteil:")
print(features_df.groupby('body_part')['is_goal'].agg(['mean', 'count']).round(3))
print()
print("Torrate nach Schusstyp:")
print(features_df.groupby('shot_type')['is_goal'].agg(['mean', 'count']).round(3))